# Lastkombinationer – M och N

| Last | M [kNm] | N [kN] | γ | ψ₀ | ξ |
|---|---:|---:|---:|---:|---:|
| Egentyngd | 150 | 100 | 1,35 | – | 0,89 |
| Trafik | 225 | 55 | 1,5 | 0,4 | – |
| Vind | 10 | 2 | 1,5 | 0,3 | – |
| Temperatur | 35 | 70 | 1,5 | 0,6 | – |

Ursprunglig 6.10a med reducerade variabla laster. γ_d = 1. Alla effekter positiva och lasterna antas samtidigt möjliga. Q₁ = trafik, Q₂ = vind, Q₃ = temperatur.

In [1]:
from pathlib import Path
import sys
import numpy as np
rot = Path.cwd().resolve()
if not (rot / "kthbygg").is_dir():
    rot = rot.parent
sys.path.insert(0, str(rot))
from kthbygg import Lastkomb

indata = dict(psi_0=[0.4, 0.3, 0.6], variabel_i_610a=True)
M = dict(G=150, Q=[225, 10, 35])
N = dict(G=100, Q=[55, 2, 70])

## Formler och beräkning

In [2]:
r_M = Lastkomb.lastkomb(**M, **indata)
r_N = Lastkomb.lastkomb(**N, **indata)
print("MOMENT")
Lastkomb.skriv_formler(**M, **indata, enhet="kNm")
print("\nNORMALKRAFT")
Lastkomb.skriv_formler(**N, **indata, enhet="kN")

MOMENT
Säkerhetsklass 3: γ_d = 1
G+ = ogynnsamma permanenta effekter; G− = gynnsamma permanenta effekter.
Q+ = max(Q, 0); gynnsamma variabla effekter utelämnas.
Alla indata ska ha samma enhet. Q numreras från 1.

6.10a: E_d,a = γ_d · 1.35 · ΣG+ + 1.00 · ΣG− + γ_d · 1.5 · Σ(ψ_0,i · Q_i+)
E_d,a = 1 · 1.35 · 150 + 1 · 1.5 · 0.4 · 225 + 1 · 1.5 · 0.3 · 10 + 1 · 1.5 · 0.6 · 35 = 373.500 [kNm]

6.10b: E_d,b,j = γ_d · 0.89 · 1.35 · ΣG+ + 1.00 · ΣG− + γ_d · 1.5 · (Q_j+ + Σ(i≠j)(ψ_0,i · Q_i+))
E_d,b (Q_1 huvudlast) = 1 · 0.89 · 1.35 · 150 + 1 · 1.5 · 225 + 1 · 1.5 · 0.3 · 10 + 1 · 1.5 · 0.6 · 35 = 553.725 [kNm]
E_d,b (Q_2 huvudlast) = 1 · 0.89 · 1.35 · 150 + 1 · 1.5 · 0.4 · 225 + 1 · 1.5 · 10 + 1 · 1.5 · 0.6 · 35 = 361.725 [kNm]
E_d,b (Q_3 huvudlast) = 1 · 0.89 · 1.35 · 150 + 1 · 1.5 · 0.4 · 225 + 1 · 1.5 · 0.3 · 10 + 1 · 1.5 · 35 = 372.225 [kNm]

E_d = max(E_d,a, E_d,b,j) = 553.725 [kNm] — dimensionerande: 6.10b, Q_1 huvudlast

NORMALKRAFT
Säkerhetsklass 3: γ_d = 1
G+ = ogynnsamma permanenta e

## Samhörande M–N-värden

Behåll värdena parvis från samma kombination vid samtidig kontroll.

In [3]:
moment = [r_M["6.10a"], *r_M["6.10b_alla"]]
normalkraft = [r_N["6.10a"], *r_N["6.10b_alla"]]
fall = ["6.10a", "6.10b trafik", "6.10b vind", "6.10b temperatur"]
print(f"{'Kombination':20} {'M [kNm]':>12} {'N [kN]':>12}")
for namn, m, kraft in zip(fall, moment, normalkraft):
    print(f"{namn:20} {m:12.3f} {kraft:12.3f}")

Kombination               M [kNm]       N [kN]
6.10a                     373.500      231.900
6.10b trafik              553.725      266.550
6.10b vind                361.725      219.150
6.10b temperatur          372.225      259.050


## Ytterligare test mot handberäknade värden

Trafik som huvudlast: M = 0,89·1,35·150 + 1,5·225 + 1,5·0,3·10 + 1,5·0,6·35 = 553,725 kNm.

N = 0,89·1,35·100 + 1,5·55 + 1,5·0,3·2 + 1,5·0,6·70 = 266,550 kN.

In [4]:
np.testing.assert_allclose(moment, [373.5, 553.725, 361.725, 372.225], rtol=0, atol=1e-9)
np.testing.assert_allclose(normalkraft, [231.9, 266.55, 219.15, 259.05], rtol=0, atol=1e-9)
for r, facit in ((r_M, 553.725), (r_N, 266.55)):
    assert r["kombination"] == "6.10b"
    assert r["huvudlast"] == 0
    np.testing.assert_allclose(r["dimensionerande"], facit, rtol=0, atol=1e-9)
print("Godkänt: alla fyra kombinationer för både M och N. Trafik styr båda.")

Godkänt: alla fyra kombinationer för både M och N. Trafik styr båda.


## Test 2 – alternativa laster från bilden

Trafik 1/2, vind 1/2 och temperatur 1/2 får inte verka samtidigt inom respektive grupp. Varje grupp får också vara frånvarande. Ursprunglig 6.10a används. Egentyngdens γ = 1,35/1 och ξ = 0,89; sättningens γ = 1,2/0 och ξ = 1 (ingen reduktion). Ingen extra säkerhetsklassfaktor.

| Last | M [kNm] | N [kN] | γ | ψ₀ |
|---|---:|---:|---:|---:|
| Egentyngd | 30 | 20 | 1,35/1 | – |
| Sättning | 10 | −5 | 1,2/0 | – |
| Trafik 1 | 25 | 10 | 1,35 | 0,4 |
| Trafik 2 | 15 | 12 | 1,35 | 0,4 |
| Vind 1 | 5 | 2 | 1,5 | 0,3 |
| Vind 2 | −5 | −2 | 1,5 | 0,3 |
| Temperatur 1 | −3 | 2 | 1,5 | 0,6 |
| Temperatur 2 | 3 | −2 | 1,5 | 0,6 |

In [5]:
from tester.test_lastkomb import G_ALTERNATIV, Q_ALTERNATIV

# Samma faktor används för både M och N från varje last.
lastfall = Lastkomb.lastkomb_610_fall(G_ALTERNATIV, Q_ALTERNATIV)
print(f"Antal kombinationer: {len(lastfall)}")

Antal kombinationer: 328


In [6]:
extremfall = {}
for namn, index, valj in (("M max", 0, max), ("M min", 0, min), ("N max", 1, max), ("N min", 1, min)):
    f = valj(lastfall, key=lambda f: f["effekt"][index])
    extremfall[namn] = f
    huvud = "ingen" if f["huvudlast"] is None else Q_ALTERNATIV[f["huvudlast"]]["namn"]
    print(f"\n{namn}: {f['kombination']}, huvudlast: {huvud}")
    print(f"M = {f['effekt'][0]:.3f} kNm; N = {f['effekt'][1]:.3f} kN")
    for index, enhet in ((0, "kNm"), (1, "kN")):
        termer = [f"{faktor:g} * ({last['effekt'][index]:g})" for faktor, last in zip(f["faktorer"], G_ALTERNATIV + Q_ALTERNATIV)]
        print(" + ".join(termer) + f" = {f['effekt'][index]:.3f} {enhet}")


M max: 6.10b, huvudlast: Trafik 1
M = 86.745 kNm; N = 30.630 kN
1.2015 * (30) + 1.2 * (10) + 1.35 * (25) + 0 * (15) + 0.45 * (5) + 0 * (-5) + 0 * (-3) + 0.9 * (3) = 86.745 kNm
1.2015 * (20) + 1.2 * (-5) + 1.35 * (10) + 0 * (12) + 0.45 * (2) + 0 * (-2) + 0 * (2) + 0.9 * (-2) = 30.630 kN

M min: 6.10b, huvudlast: Vind 2
M = 19.800 kNm; N = 18.800 kN
1 * (30) + 0 * (10) + 0 * (25) + 0 * (15) + 0 * (5) + 1.5 * (-5) + 0.9 * (-3) + 0 * (3) = 19.800 kNm
1 * (20) + 0 * (-5) + 0 * (10) + 0 * (12) + 0 * (2) + 1.5 * (-2) + 0.9 * (2) + 0 * (-2) = 18.800 kN

N max: 6.10b, huvudlast: Trafik 2
M = 55.845 kNm; N = 42.930 kN
1.2015 * (30) + 0 * (10) + 0 * (25) + 1.35 * (15) + 0.45 * (5) + 0 * (-5) + 0.9 * (-3) + 0 * (3) = 55.845 kNm
1.2015 * (20) + 0 * (-5) + 0 * (10) + 1.35 * (12) + 0.45 * (2) + 0 * (-2) + 0.9 * (2) + 0 * (-2) = 42.930 kN

N min: 6.10b, huvudlast: Vind 2
M = 37.200 kNm; N = 9.200 kN
1 * (30) + 1.2 * (10) + 0 * (25) + 0 * (15) + 0 * (5) + 1.5 * (-5) + 0 * (-3) + 0.9 * (3) = 37.200 kNm

### Kontroll

Max M och max N uppstår i olika kombinationer. Använd respektive samhörande M–N-par vid interaktionskontroll; för en allmän interaktionskontroll finns samtliga par i `lastfall`.

In [7]:
np.testing.assert_allclose(extremfall["M max"]["effekt"], [86.745, 30.63], rtol=0, atol=1e-9)
np.testing.assert_allclose(extremfall["N max"]["effekt"], [55.845, 42.93], rtol=0, atol=1e-9)
np.testing.assert_allclose(extremfall["M min"]["effekt"][0], 19.8)
np.testing.assert_allclose(extremfall["N min"]["effekt"][1], 9.2)
for f in lastfall:
    for i, j in ((2, 3), (4, 5), (6, 7)):
        assert f["faktorer"][i] * f["faktorer"][j] == 0
print("Godkänt: extremvärden, samhörande effekter och ömsesidigt uteslutande laster.")

Godkänt: extremvärden, samhörande effekter och ömsesidigt uteslutande laster.
